### Title of Project: Cryptocurrency Time-Series Analysis and Prediction Using Deep Learning

##### Data analyst: Hamed Ahmadinia

##### Arcada University of Applied Sciences

##### Data source: Historical cryptocurrency market data collected via public APIs (e.g., Binance, CoinGecko)

##### Computing environment: CSC Puhti supercomputing environment

##### Start of Project: 15.3.2026
##### End of Project: 24.3.2026

###### Email: hamed.ahmadinia@arcada.fi
###### Web: www.ahmadinia.fi
###### © 2026 Hamed Ahmadinia – CC BY-NC 4.0

### 1. Imports

In [1]:
import os
import time
import math
import json
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### 2. Configuration

In [2]:
PROJECT_ROOT = "crypto_project"
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
LOG_DIR = os.path.join(PROJECT_ROOT, "logs")

for path in [RAW_DIR, PROCESSED_DIR, MODEL_DIR, RESULTS_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(path, exist_ok=True)

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "SOLUSDT",
    "XRPUSDT",
    "ADAUSDT",
    "DOGEUSDT",
    "MATICUSDT",
    "DOTUSDT",
    "LTCUSDT"
]

# For first test on Puhti, you may temporarily use:
# SYMBOLS = ["BTCUSDT", "ETHUSDT"]

INTERVAL = "1m"          # "1h", "15m", "5m", "1m"
START_DATE = "2021-01-01"
SEQ_LEN = 48
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
TRAIN_RATIO = 0.8
SLEEP_SECONDS = 0.20

MAX_PROVIDER_RETRIES = 4
MAX_COIN_FAILURES = 20
CHUNK_SAVE_EVERY = 20

### 3. GPU check

In [3]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.9.1+cu129
CUDA available: True
Using device: cuda
GPU: Tesla V100-SXM2-32GB


### 4. Helpers: symbol mapping

In [4]:
def binance_symbol(symbol):
    return symbol

def kucoin_symbol(symbol):
    if symbol.endswith("USDT"):
        return symbol[:-4] + "-USDT"
    return symbol

def coinbase_symbol(symbol):
    # Coinbase often uses USD instead of USDT
    if symbol.endswith("USDT"):
        return symbol[:-4] + "-USD"
    return None

### 5. Logging and checkpoint utils

In [5]:
def checkpoint_path(symbol):
    return os.path.join(CHECKPOINT_DIR, f"{symbol}_{INTERVAL}_checkpoint.json")

def raw_path(symbol):
    return os.path.join(RAW_DIR, f"{symbol}_{INTERVAL}.parquet")

def log_path():
    return os.path.join(LOG_DIR, f"download_log_{INTERVAL}.txt")

def write_log(message):
    ts = pd.Timestamp.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{ts}] {message}"
    print(line)
    with open(log_path(), "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_checkpoint(symbol, next_start_time, provider_name, rows_downloaded):
    payload = {
        "symbol": symbol,
        "interval": INTERVAL,
        "next_start_time": int(next_start_time) if next_start_time is not None else None,
        "provider_name": provider_name,
        "rows_downloaded": int(rows_downloaded),
        "updated_at": pd.Timestamp.utcnow().isoformat()
    }
    with open(checkpoint_path(symbol), "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def load_checkpoint(symbol):
    fp = checkpoint_path(symbol)
    if not os.path.exists(fp):
        return None
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)

def get_resume_start_time(symbol):
    cp = load_checkpoint(symbol)
    if cp and cp.get("next_start_time") is not None:
        return int(cp["next_start_time"])
    return int(pd.Timestamp(START_DATE).timestamp() * 1000)

### 6. Standard schema normalizer

In [6]:
STANDARD_COLUMNS = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
    "ignore", "symbol", "interval", "provider"
]

def normalize_ohlcv_df(df, symbol, interval, provider_name):
    expected_cols = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    df = df[expected_cols].copy()

    numeric_cols = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", errors="coerce")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", errors="coerce")
    df["symbol"] = symbol
    df["interval"] = interval
    df["provider"] = provider_name

    return df[STANDARD_COLUMNS]

### 7. Provider 1: Binance

In [7]:
BINANCE_URL = "https://api.binance.com/api/v3/klines"

def fetch_binance(symbol, interval="1m", start_time=None, limit=1000, timeout=60):
    params = {
        "symbol": binance_symbol(symbol),
        "interval": interval,
        "limit": limit
    }
    if start_time is not None:
        params["startTime"] = int(start_time)

    response = requests.get(BINANCE_URL, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    if not data:
        return pd.DataFrame()

    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df = pd.DataFrame(data, columns=columns)
    return normalize_ohlcv_df(df, symbol, interval, "binance")

### 8. Provider 2: KuCoin

In [8]:
KUCOIN_URL = "https://api.kucoin.com/api/v1/market/candles"

def kucoin_interval(interval):
    mapping = {
        "1m": "1min",
        "5m": "5min",
        "15m": "15min",
        "1h": "1hour"
    }
    return mapping.get(interval, None)

def interval_millis(interval):
    mapping = {
        "1m": 60_000,
        "5m": 300_000,
        "15m": 900_000,
        "1h": 3_600_000
    }
    return mapping.get(interval, 60_000)

def fetch_kucoin(symbol, interval="1m", start_time=None, limit=1000, timeout=60):
    kc_interval = kucoin_interval(interval)
    kc_symbol = kucoin_symbol(symbol)

    if kc_interval is None or kc_symbol is None:
        return pd.DataFrame()

    params = {
        "type": kc_interval,
        "symbol": kc_symbol
    }

    response = requests.get(KUCOIN_URL, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()

    if "data" not in payload or not payload["data"]:
        return pd.DataFrame()

    rows = payload["data"][:limit]
    rows = list(reversed(rows))

    step_ms = interval_millis(interval)

    parsed = []
    for row in rows:
        # [time, open, close, high, low, volume, turnover]
        ts_sec, open_, close_, high_, low_, volume_, turnover_ = row
        open_ms = int(ts_sec) * 1000
        close_ms = open_ms + step_ms - 1

        parsed.append([
            open_ms, open_, high_, low_, close_, volume_,
            close_ms, turnover_, np.nan, np.nan, np.nan, np.nan
        ])

    df = pd.DataFrame(parsed, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ])

    if start_time is not None:
        df = df[df["open_time"] >= int(start_time)].copy()

    return normalize_ohlcv_df(df, symbol, interval, "kucoin")

### 9. Provider 3: Coinbase

In [9]:
COINBASE_URL_TEMPLATE = "https://api.exchange.coinbase.com/products/{product_id}/candles"

def coinbase_granularity(interval):
    mapping = {
        "1m": 60,
        "5m": 300,
        "15m": 900,
        "1h": 3600
    }
    return mapping.get(interval, None)

def fetch_coinbase(symbol, interval="1m", start_time=None, limit=300, timeout=60):
    product_id = coinbase_symbol(symbol)
    granularity = coinbase_granularity(interval)

    if product_id is None or granularity is None:
        return pd.DataFrame()

    url = COINBASE_URL_TEMPLATE.format(product_id=product_id)
    params = {"granularity": granularity}

    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    if not data or isinstance(data, dict):
        return pd.DataFrame()

    parsed = []
    for row in data:
        # [time, low, high, open, close, volume]
        ts_sec, low_, high_, open_, close_, volume_ = row
        open_ms = int(ts_sec) * 1000
        close_ms = open_ms + granularity * 1000 - 1
        parsed.append([
            open_ms, open_, high_, low_, close_, volume_,
            close_ms, np.nan, np.nan, np.nan, np.nan, np.nan
        ])

    df = pd.DataFrame(parsed, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ])

    df = df.sort_values("open_time").reset_index(drop=True)

    if start_time is not None:
        df = df[df["open_time"] >= int(start_time)].copy()

    return normalize_ohlcv_df(df, symbol, interval, "coinbase")

### 10. Provider registry

In [10]:
PROVIDERS = [
    ("binance", fetch_binance),
    ("kucoin", fetch_kucoin),
    ("coinbase", fetch_coinbase),
]

### 11. Parquet append/save

In [11]:
def append_and_save(symbol, df_chunk):
    fp = raw_path(symbol)

    if os.path.exists(fp):
        existing = pd.read_parquet(fp)
        combined = pd.concat([existing, df_chunk], ignore_index=True)
        combined = combined.drop_duplicates(subset=["open_time", "symbol"]).sort_values("open_time")
    else:
        combined = df_chunk.drop_duplicates(subset=["open_time", "symbol"]).sort_values("open_time")

    combined.to_parquet(fp, index=False)
    return combined

### 12. Provider rotation fetcher

In [12]:
def fetch_from_any_provider(symbol, interval, start_time, limit=1000):
    provider_errors = []

    for provider_name, provider_fn in PROVIDERS:
        for attempt in range(MAX_PROVIDER_RETRIES):
            try:
                df = provider_fn(
                    symbol=symbol,
                    interval=interval,
                    start_time=start_time,
                    limit=limit
                )

                if df is not None and not df.empty:
                    return df, provider_name

                provider_errors.append(f"{provider_name}: empty response")
                break

            except Exception as e:
                wait = min(2 ** attempt + random.random(), 15)
                provider_errors.append(f"{provider_name} attempt {attempt+1}: {e}")
                write_log(f"{symbol} | provider={provider_name} failed | attempt={attempt+1} | error={e}")
                time.sleep(wait)

    return pd.DataFrame(), provider_errors

### 13. Robust per-coin downloader

In [13]:
def download_coin_robust(symbol, interval="1m", sleep_seconds=0.2, chunk_save_every=20):
    write_log(f"START coin={symbol}")

    start_ts = get_resume_start_time(symbol)
    total_downloaded = 0
    chunk_counter = 0
    failure_counter = 0
    finished = False

    while not finished:
        df_chunk, provider_info = fetch_from_any_provider(
            symbol=symbol,
            interval=interval,
            start_time=start_ts,
            limit=1000
        )

        if df_chunk is None or df_chunk.empty:
            failure_counter += 1
            write_log(f"{symbol} | no usable chunk | failures={failure_counter} | providers={provider_info}")

            if failure_counter >= MAX_COIN_FAILURES:
                write_log(f"{symbol} | STOP after repeated failures")
                break

            time.sleep(5)
            continue

        failure_counter = 0

        df_chunk = df_chunk.dropna(subset=["open_time"]).copy()
        if df_chunk.empty:
            failure_counter += 1
            continue

        last_open_time = int(df_chunk["open_time"].max().timestamp() * 1000)
        next_start = last_open_time + 1

        append_and_save(symbol, df_chunk)

        total_downloaded += len(df_chunk)
        chunk_counter += 1

        provider_used = provider_info
        save_checkpoint(
            symbol=symbol,
            next_start_time=next_start,
            provider_name=provider_used,
            rows_downloaded=total_downloaded
        )

        if chunk_counter % chunk_save_every == 0:
            write_log(f"{symbol} | provider={provider_used} | downloaded_rows={total_downloaded:,}")

        if len(df_chunk) < 1000:
            write_log(f"{symbol} | final partial chunk received | provider={provider_used}")
            finished = True
        else:
            start_ts = next_start
            time.sleep(sleep_seconds)

    fp = raw_path(symbol)
    if os.path.exists(fp):
        final_df = pd.read_parquet(fp).sort_values("open_time").reset_index(drop=True)
        write_log(f"END coin={symbol} | final_rows={len(final_df):,}")
        return final_df

    write_log(f"END coin={symbol} | no file created")
    return pd.DataFrame()

### 14. Download all coins

In [ ]:
all_downloaded = []
failed_symbols = []

for symbol in SYMBOLS:
    try:
        df_symbol = download_coin_robust(
            symbol=symbol,
            interval=INTERVAL,
            sleep_seconds=SLEEP_SECONDS,
            chunk_save_every=CHUNK_SAVE_EVERY
        )

        if df_symbol.empty:
            failed_symbols.append(symbol)
            write_log(f"{symbol} | marked as failed")
            continue

        all_downloaded.append(df_symbol)
        write_log(f"{symbol} | completed successfully")

    except Exception as e:
        failed_symbols.append(symbol)
        write_log(f"{symbol} | crashed unexpectedly | error={e}")
        continue

if all_downloaded:
    df_all = pd.concat(all_downloaded, ignore_index=True)
    df_all = df_all.drop_duplicates(subset=["open_time", "symbol"]).sort_values(["symbol", "open_time"])
    print("Combined shape:", df_all.shape)
else:
    df_all = pd.DataFrame()
    print("No successful downloads.")

print("Failed symbols:", failed_symbols)

[2026-03-25 16:55:19 UTC] START coin=BTCUSDT
[2026-03-25 16:55:22 UTC] BTCUSDT | final partial chunk received | provider=binance
[2026-03-25 16:55:23 UTC] END coin=BTCUSDT | final_rows=2,748,903
[2026-03-25 16:55:23 UTC] BTCUSDT | completed successfully
[2026-03-25 16:55:23 UTC] START coin=ETHUSDT
[2026-03-25 16:55:27 UTC] ETHUSDT | final partial chunk received | provider=binance
[2026-03-25 16:55:28 UTC] END coin=ETHUSDT | final_rows=2,748,903
[2026-03-25 16:55:28 UTC] ETHUSDT | completed successfully
[2026-03-25 16:55:28 UTC] START coin=BNBUSDT
[2026-03-25 16:55:32 UTC] BNBUSDT | final partial chunk received | provider=binance
[2026-03-25 16:55:33 UTC] END coin=BNBUSDT | final_rows=2,748,903
[2026-03-25 16:55:33 UTC] BNBUSDT | completed successfully
[2026-03-25 16:55:33 UTC] START coin=SOLUSDT
[2026-03-25 16:55:38 UTC] SOLUSDT | final partial chunk received | provider=binance
[2026-03-25 16:55:38 UTC] END coin=SOLUSDT | final_rows=2,748,904
[2026-03-25 16:55:38 UTC] SOLUSDT | complet

### 15. Raw size check

In [ ]:
raw_files = [raw_path(symbol) for symbol in SYMBOLS if os.path.exists(raw_path(symbol))]
total_size_bytes = sum(os.path.getsize(fp) for fp in raw_files) 
total_size_gb = total_size_bytes / (1024 ** 3)
print(f"Approx raw parquet size: {total_size_gb:.3f} GB")

### 16. Feature engineering

In [ ]:
def add_features(df_symbol):
    df_symbol = df_symbol.sort_values("open_time").copy()

    df_symbol["return_1"] = df_symbol["close"].pct_change()
    df_symbol["log_return_1"] = np.log(df_symbol["close"] / df_symbol["close"].shift(1))

    df_symbol["ma_24"] = df_symbol["close"].rolling(24).mean()
    df_symbol["std_24"] = df_symbol["close"].rolling(24).std()

    df_symbol["ma_72"] = df_symbol["close"].rolling(72).mean()
    df_symbol["std_72"] = df_symbol["close"].rolling(72).std()

    df_symbol["ma_168"] = df_symbol["close"].rolling(168).mean()
    df_symbol["std_168"] = df_symbol["close"].rolling(168).std()

    df_symbol["volume_ma_24"] = df_symbol["volume"].rolling(24).mean()

    df_symbol = df_symbol.dropna().reset_index(drop=True)
    return df_symbol


feature_dfs = []

for symbol in SYMBOLS:
    fp = raw_path(symbol)
    if not os.path.exists(fp):
        print(f"Missing raw file for {symbol}, skipping.")
        continue

    coin_df = pd.read_parquet(fp)
    if coin_df.empty:
        print(f"Empty raw file for {symbol}, skipping.")
        continue

    coin_df = add_features(coin_df)
    if coin_df.empty:
        print(f"No usable data after feature engineering for {symbol}, skipping.")
        continue

    feature_dfs.append(coin_df)

if len(feature_dfs) == 0:
    raise ValueError("No usable feature data was produced.")

df_features = pd.concat(feature_dfs, ignore_index=True)
df_features = df_features.sort_values(["symbol", "open_time"]).reset_index(drop=True)

processed_path = os.path.join(PROCESSED_DIR, f"all_coins_features_{INTERVAL}.parquet")
df_features.to_parquet(processed_path, index=False)

print("Saved processed features:", processed_path)
print("Processed shape:", df_features.shape)
print(df_features.head())

### 17. Feature selection

In [ ]:
feature_cols = [
    "open", "high", "low", "close", "volume", "number_of_trades",
    "quote_asset_volume", "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
    "return_1", "log_return_1",
    "ma_24", "std_24",
    "ma_72", "std_72",
    "ma_168", "std_168",
    "volume_ma_24"
]

target_col = "close"
target_index = feature_cols.index(target_col)

### 18. Sequence creation with labels

In [ ]:
def create_sequences_with_labels(data_array, target_index, seq_len=48, symbol_name=None):
    X, y, labels = [], [], []

    for i in range(seq_len, len(data_array)):
        X.append(data_array[i-seq_len:i])
        y.append(data_array[i, target_index])
        labels.append(symbol_name)

    return np.array(X), np.array(y), np.array(labels)


all_train_X, all_train_y, all_train_labels = [], [], []
all_test_X, all_test_y, all_test_labels = [], [], []
scalers = {}
usable_symbols = []

for symbol in sorted(df_features["symbol"].unique()):
    coin_df = df_features[df_features["symbol"] == symbol].copy()
    coin_data = coin_df[feature_cols].copy()

    if len(coin_data) < SEQ_LEN + 20:
        print(f"Not enough data for {symbol}, skipping.")
        continue

    train_size = int(len(coin_data) * TRAIN_RATIO)
    train_df = coin_data.iloc[:train_size].copy()
    test_df = coin_data.iloc[train_size:].copy()

    if len(train_df) < SEQ_LEN + 1 or len(test_df) < SEQ_LEN + 1:
        print(f"Train/test split too small for {symbol}, skipping.")
        continue

    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_df)
    test_scaled = scaler.transform(test_df)
    scalers[symbol] = scaler

    X_train_coin, y_train_coin, labels_train_coin = create_sequences_with_labels(
        train_scaled,
        target_index=target_index,
        seq_len=SEQ_LEN,
        symbol_name=symbol
    )

    X_test_coin, y_test_coin, labels_test_coin = create_sequences_with_labels(
        test_scaled,
        target_index=target_index,
        seq_len=SEQ_LEN,
        symbol_name=symbol
    )

    if len(X_train_coin) == 0 or len(X_test_coin) == 0:
        print(f"No sequences for {symbol}, skipping.")
        continue

    all_train_X.append(X_train_coin)
    all_train_y.append(y_train_coin)
    all_train_labels.append(labels_train_coin)

    all_test_X.append(X_test_coin)
    all_test_y.append(y_test_coin)
    all_test_labels.append(labels_test_coin)

    usable_symbols.append(symbol)

if len(all_train_X) == 0:
    raise ValueError("No training sequences were created.")

X_train = np.concatenate(all_train_X, axis=0)
y_train = np.concatenate(all_train_y, axis=0)
train_labels = np.concatenate(all_train_labels, axis=0)

X_test = np.concatenate(all_test_X, axis=0)
y_test = np.concatenate(all_test_y, axis=0)
test_labels = np.concatenate(all_test_labels, axis=0)

print("Usable symbols:", usable_symbols)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("train_labels:", train_labels.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("test_labels:", test_labels.shape)

### 19. Dataset / DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 20. LSTM model

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

### 21. Transformer model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


class TransformerRegressor(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

### 22. Training / evaluation helpers

In [ ]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=10):
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history.append(epoch_loss)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.6f}")

    return history


def evaluate_model_with_arrays(model, test_loader, device):
    model.eval()
    preds_list = []
    true_list = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy().flatten()
            preds_list.extend(preds)
            true_list.extend(y_batch.numpy().flatten())

    preds_arr = np.array(preds_list)
    true_arr = np.array(true_list)

    mae = mean_absolute_error(true_arr, preds_arr)
    rmse = math.sqrt(mean_squared_error(true_arr, preds_arr))

    return preds_arr, true_arr, mae, rmse


def inverse_transform_target_per_coin(scaled_values, scaler, target_index, n_features):
    dummy = np.zeros((len(scaled_values), n_features))
    dummy[:, target_index] = scaled_values
    restored = scaler.inverse_transform(dummy)
    return restored[:, target_index]

### 23. Train LSTM

In [ ]:
lstm_model = LSTMRegressor(input_size=len(feature_cols)).to(device)
criterion = nn.MSELoss()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)

print("\nTraining LSTM...")
lstm_history = train_model(
    model=lstm_model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=lstm_optimizer,
    device=device,
    epochs=EPOCHS
)

lstm_preds, y_true, lstm_mae, lstm_rmse = evaluate_model_with_arrays(lstm_model, test_loader, device)
print("LSTM scaled MAE:", lstm_mae)
print("LSTM scaled RMSE:", lstm_rmse)

### 24. Train Transformer

In [ ]:
transformer_model = TransformerRegressor(
    input_size=len(feature_cols),
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1
).to(device)

criterion = nn.MSELoss()
transformer_optimizer = torch.optim.Adam(transformer_model.parameters(), lr=LEARNING_RATE)

print("\nTraining Transformer...")
transformer_history = train_model(
    model=transformer_model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=transformer_optimizer,
    device=device,
    epochs=EPOCHS
)

transformer_preds, y_true_2, transformer_mae, transformer_rmse = evaluate_model_with_arrays(
    transformer_model, test_loader, device
)

print("Transformer scaled MAE:", transformer_mae)
print("Transformer scaled RMSE:", transformer_rmse)

### 25. Overall comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["LSTM", "Transformer"],
    "Scaled_MAE": [lstm_mae, transformer_mae],
    "Scaled_RMSE": [lstm_rmse, transformer_rmse]
})

print("\nOverall comparison:")
print(comparison_df)

### 26. Per-coin evaluation (scaled)

In [ ]:
per_coin_results = []

for symbol in usable_symbols:
    mask = (test_labels == symbol)

    y_coin = y_true[mask]
    lstm_coin = lstm_preds[mask]
    transformer_coin = transformer_preds[mask]

    lstm_mae_coin = mean_absolute_error(y_coin, lstm_coin)
    lstm_rmse_coin = math.sqrt(mean_squared_error(y_coin, lstm_coin))

    transformer_mae_coin = mean_absolute_error(y_coin, transformer_coin)
    transformer_rmse_coin = math.sqrt(mean_squared_error(y_coin, transformer_coin))

    better_model = "LSTM" if lstm_rmse_coin < transformer_rmse_coin else "Transformer"

    per_coin_results.append({
        "Coin": symbol.replace("USDT", ""),
        "Num_Test_Samples": int(mask.sum()),
        "LSTM_MAE": lstm_mae_coin,
        "LSTM_RMSE": lstm_rmse_coin,
        "Transformer_MAE": transformer_mae_coin,
        "Transformer_RMSE": transformer_rmse_coin,
        "Better_Model_By_RMSE": better_model
    })

per_coin_df = pd.DataFrame(per_coin_results).sort_values("Coin").reset_index(drop=True)

print("\nPer-coin comparison (scaled):")
print(per_coin_df.round(6))

### 27. Per-coin evaluation (original scale)

In [ ]:
per_coin_original_results = []

for symbol in usable_symbols:
    mask = (test_labels == symbol)

    y_coin_scaled = y_true[mask]
    lstm_coin_scaled = lstm_preds[mask]
    transformer_coin_scaled = transformer_preds[mask]

    scaler = scalers[symbol]

    y_coin_orig = inverse_transform_target_per_coin(
        y_coin_scaled, scaler, target_index, len(feature_cols)
    )
    lstm_coin_orig = inverse_transform_target_per_coin(
        lstm_coin_scaled, scaler, target_index, len(feature_cols)
    )
    transformer_coin_orig = inverse_transform_target_per_coin(
        transformer_coin_scaled, scaler, target_index, len(feature_cols)
    )

    lstm_mae_orig = mean_absolute_error(y_coin_orig, lstm_coin_orig)
    lstm_rmse_orig = math.sqrt(mean_squared_error(y_coin_orig, lstm_coin_orig))

    transformer_mae_orig = mean_absolute_error(y_coin_orig, transformer_coin_orig)
    transformer_rmse_orig = math.sqrt(mean_squared_error(y_coin_orig, transformer_coin_orig))

    better_model = "LSTM" if lstm_rmse_orig < transformer_rmse_orig else "Transformer"

    per_coin_original_results.append({
        "Coin": symbol.replace("USDT", ""),
        "LSTM_MAE_Original": lstm_mae_orig,
        "LSTM_RMSE_Original": lstm_rmse_orig,
        "Transformer_MAE_Original": transformer_mae_orig,
        "Transformer_RMSE_Original": transformer_rmse_orig,
        "Better_Model_By_Original_RMSE": better_model
    })

per_coin_original_df = pd.DataFrame(per_coin_original_results).sort_values("Coin").reset_index(drop=True)

print("\nPer-coin comparison (original price scale):")
print(per_coin_original_df.round(4))

### 28. Save outputs

In [ ]:
torch.save(lstm_model.state_dict(), os.path.join(MODEL_DIR, f"lstm_multi_coin_{INTERVAL}.pt"))
torch.save(transformer_model.state_dict(), os.path.join(MODEL_DIR, f"transformer_multi_coin_{INTERVAL}.pt"))

comparison_df.to_csv(os.path.join(RESULTS_DIR, f"model_comparison_{INTERVAL}.csv"), index=False)
per_coin_df.to_csv(os.path.join(RESULTS_DIR, f"per_coin_model_comparison_scaled_{INTERVAL}.csv"), index=False)
per_coin_original_df.to_csv(os.path.join(RESULTS_DIR, f"per_coin_model_comparison_original_{INTERVAL}.csv"), index=False)

print("\nSaved models and result tables.")

### 29. Plot training loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(lstm_history, label="LSTM")
plt.plot(transformer_history, label="Transformer")
plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

### 30. Plot overall predictions

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_true[:500], label="True", alpha=0.8)
plt.plot(lstm_preds[:500], label="LSTM Pred", alpha=0.8)
plt.plot(transformer_preds[:500], label="Transformer Pred", alpha=0.8)
plt.title("Prediction Comparison (Scaled)")
plt.xlabel("Time Step")
plt.ylabel("Scaled Close Price")
plt.legend()
plt.tight_layout()
plt.show()

### 31. Plot per-coin RMSE

In [ ]:
plot_df = per_coin_df.set_index("Coin")[["LSTM_RMSE", "Transformer_RMSE"]]
plot_df.plot(kind="bar", figsize=(10, 5))
plt.title("Per-Coin RMSE Comparison (Scaled)")
plt.ylabel("RMSE")
plt.xlabel("Coin")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 32. Report-friendly tables

In [ ]:
report_table_scaled = per_coin_df.copy()
for col in ["LSTM_MAE", "LSTM_RMSE", "Transformer_MAE", "Transformer_RMSE"]:
    report_table_scaled[col] = report_table_scaled[col].round(6)

report_table_original = per_coin_original_df.copy()
for col in ["LSTM_MAE_Original", "LSTM_RMSE_Original", "Transformer_MAE_Original", "Transformer_RMSE_Original"]:
    report_table_original[col] = report_table_original[col].round(4)

print("\nReport table (scaled):")
print(report_table_scaled)

print("\nReport table (original scale):")
print(report_table_original)